In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def get_norm(num_channels, norm_type="batch", num_groups=8):
    norm_type = norm_type.lower()

    if norm_type in ["batch", "bn", "batchnorm"]:
        return nn.BatchNorm2d(num_channels)

    elif norm_type in ["group", "gn", "groupnorm"]:
        print("using gn")
        g = min(num_groups, num_channels)
        while num_channels % g != 0 and g > 1:
            g -= 1
        return nn.GroupNorm(g, num_channels)

    elif norm_type in ["instance", "in", "instancenorm"]:
        return nn.InstanceNorm2d(num_channels, affine=True)

    else:
        raise ValueError(f"Normalized no supported: {norm_type}")


# ------------------------------
# Attention (SE)
# ------------------------------
class SEBlockV2(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        hidden = max(1, ch // r)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(ch, hidden, 1, bias=True),
            nn.GELU(),
            nn.Conv2d(hidden, ch, 1, bias=True),
            nn.Sigmoid()
        )

    def forward(self, x):
        s = self.fc(self.pool(x))
        return x * s


# ------------------------------
# Conv + Norm + Act (+ optional SE)
# ------------------------------
class ConvNormActV2(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, act=True, se=False,
                 norm_type="batch", num_groups=8):
        super().__init__()
        self.pad = nn.ReflectionPad2d(p) if p > 0 else nn.Identity()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=0, bias=False)
        self.norm = get_norm(out_ch, norm_type=norm_type, num_groups=num_groups)
        self.act  = nn.GELU() if act else nn.Identity()
        self.se   = SEBlockV2(out_ch) if se else nn.Identity()

    def forward(self, x):
        x = self.pad(x)
        x = self.conv(x)
        x = self.norm(x)
        x = self.act(x)
        x = self.se(x)
        return x


# ------------------------------
# Residual block (dilated)
# ------------------------------
class ResBlockV2(nn.Module):
    def __init__(self, ch, dilation=1, se=False, norm_type="group", num_groups=8):
        super().__init__()

        self.c1 = ConvNormActV2(
            ch, ch, k=3, s=1, p=dilation,
            act=True, se=se, norm_type=norm_type, num_groups=num_groups
        )
        self.c2 = ConvNormActV2(
            ch, ch, k=3, s=1, p=dilation,
            act=False, se=False, norm_type=norm_type, num_groups=num_groups
        )

        self.c1.conv.dilation = (dilation, dilation)
        self.c2.conv.dilation = (dilation, dilation)

    def forward(self, x):
        return x + self.c2(self.c1(x))


# ------------------------------
# Downsample block
# ------------------------------
class DownV2(nn.Module):
    def __init__(self, in_ch, out_ch, se=True, norm_type="group", num_groups=8):
        super().__init__()
        self.block = nn.Sequential(
            ConvNormActV2(
                in_ch, out_ch, k=3, s=2, p=1,
                act=True, se=se, norm_type=norm_type, num_groups=num_groups
            ),
            ConvNormActV2(
                out_ch, out_ch, k=3, s=1, p=1,
                act=True, se=se, norm_type=norm_type, num_groups=num_groups
            )
        )

    def forward(self, x):
        return self.block(x)


# ------------------------------
# ASPP
# ------------------------------
class ASPPV2(nn.Module):
    def __init__(self, ch, rates=(1, 2, 4, 8), se=True, norm_type="group", num_groups=8):
        super().__init__()

        self.branches = nn.ModuleList([
            ConvNormActV2(
                ch, ch, k=3, s=1, p=r,
                act=True, se=False, norm_type=norm_type, num_groups=num_groups
            ) for r in rates
        ])

        for b, r in zip(self.branches, rates):
            b.conv.dilation = (r, r)

        self.fuse = ConvNormActV2(
            ch * len(rates), ch, k=1, s=1, p=0,
            act=True, se=se, norm_type=norm_type, num_groups=num_groups
        )

    def forward(self, x):
        xs = [b(x) for b in self.branches]
        x = torch.cat(xs, dim=1)
        return self.fuse(x)


# ------------------------------
# Upsample + concat skip
# ------------------------------
class UpCatV2(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, se=True, norm_type="group", num_groups=8):
        super().__init__()
        self.conv = nn.Sequential(
            ConvNormActV2(
                in_ch + skip_ch, out_ch, k=3, s=1, p=1,
                act=True, se=se, norm_type=norm_type, num_groups=num_groups
            ),
            ConvNormActV2(
                out_ch, out_ch, k=3, s=1, p=1,
                act=True, se=se, norm_type=norm_type, num_groups=num_groups
            ),
        )

    def forward(self, x, skip):
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


# ------------------------------
# Generator V2
# ------------------------------
class ResUNetLLIEV2(nn.Module):
    def __init__(self, base=48, n_res=6, aspp_rates=(1, 2, 4, 8), norm_type="group", num_groups=8):
        super().__init__()

        self.in0 = nn.Sequential(
            ConvNormActV2(3, base, k=7, s=1, p=3, se=True, norm_type=norm_type, num_groups=num_groups),
            ConvNormActV2(base, base, k=3, s=1, p=1, se=True, norm_type=norm_type, num_groups=num_groups)
        )

        self.d1 = DownV2(base, base * 2, se=True, norm_type=norm_type, num_groups=num_groups)
        self.d2 = DownV2(base * 2, base * 4, se=True, norm_type=norm_type, num_groups=num_groups)
        self.d3 = DownV2(base * 4, base * 8, se=True, norm_type=norm_type, num_groups=num_groups)

        res_blocks = []
        for i in range(n_res):
            dilation = 1 if i < n_res // 2 else 2
            res_blocks.append(
                ResBlockV2(base * 8, dilation=dilation, se=False,
                           norm_type=norm_type, num_groups=num_groups)
            )
        self.res = nn.Sequential(*res_blocks)

        self.aspp = ASPPV2(base * 8, rates=aspp_rates, se=True,
                           norm_type=norm_type, num_groups=num_groups)

        self.u2 = UpCatV2(base * 8, base * 4, base * 4, se=True, norm_type=norm_type, num_groups=num_groups)
        self.u1 = UpCatV2(base * 4, base * 2, base * 2, se=True, norm_type=norm_type, num_groups=num_groups)
        self.u0 = UpCatV2(base * 2, base,     base,     se=True, norm_type=norm_type, num_groups=num_groups)

        self.out = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(base, 3, kernel_size=3, stride=1, padding=0),
        )

    def forward(self, x):
        x0 = self.in0(x)
        x1 = self.d1(x0)
        x2 = self.d2(x1)
        x3 = self.d3(x2)

        b = self.aspp(self.res(x3))

        u2 = self.u2(b, x2)
        u1 = self.u1(u2, x1)
        u0 = self.u0(u1, x0)

        r = torch.tanh(self.out(u0)) * 0.5
        y = torch.clamp(x + r, 0.0, 1.0)
        return y


# ------------------------------
# Conditional PatchGAN Discriminator V2
# ------------------------------
class PatchDiscriminatorSNCondV2(nn.Module):
    def __init__(self, base=64):
        super().__init__()
        sn = nn.utils.spectral_norm

        def block(in_ch, out_ch, s, norm=True):
            layers = [sn(nn.Conv2d(in_ch, out_ch, kernel_size=4, stride=s, padding=1))]
            if norm:
                layers.append(nn.BatchNorm2d(out_ch))   # default: BatchNorm
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return nn.Sequential(*layers)

        self.net = nn.Sequential(
            block(6,       base,   2, norm=False),  # /2
            block(base,    base*2, 2, norm=True),   # /4
            block(base*2,  base*4, 2, norm=True),   # /8
            block(base*4,  base*8, 2, norm=True),   # /16
            sn(nn.Conv2d(base*8, 1, kernel_size=4, stride=1, padding=1))
        )

    def forward(self, x_low, y_img):
        inp = torch.cat([x_low, y_img], dim=1)
        return self.net(inp)

In [3]:
#import inspect
#print(inspect.signature(DownV2.__init__))
#print(inspect.signature(ResUNetLLIEV2.__init__))

(self, in_ch, out_ch, se=True, norm_type='batch', num_groups=8)
(self, base=48, n_res=6, aspp_rates=(1, 2, 4, 8), norm_type='batch', num_groups=8)
